# GitHub API / CLI 全流程验证

> **目标**：验证 GitHub REST API 的完整能力链路，为 Agent 工具开发提供依据
> **覆盖**：认证 → 仓库 → 内容 → Issue → PR → 搜索 → 提交 → 工作流
> **认证**：`GITHUB_TOKEN`（Personal Access Token）

每个 API 调用旁标注等效的 `gh` CLI 命令，供对比参考。

In [ ]:
from github_client import GitHubClient, format_repo_info, format_issue, format_pull, format_commit
from pathlib import Path
import json

# 初始化客户端（自动从 .env 读取 GITHUB_TOKEN）
client = GitHubClient()

# 健康检查
health = client.health_check()
print(f"[{'OK' if health['ok'] else 'FAIL'}] GitHubClient initialized")
if health['ok']:
    print(f"       User: @{health['login']}")
else:
    print(f"       Error: {health.get('error')}")
    print(f"       请检查 .env 中的 GITHUB_TOKEN 是否有效")

In [ ]:
a= 1
a

## 1. 仓库操作

验证 `GET /repos/{owner}/{repo}` 和搜索能力。

In [ ]:
# 1.1 获取仓库信息
# gh> gh repo view facebook/react --json name,description,stargazersCount
repo = client.get_repo("facebook", "react")
print(format_repo_info(repo))

In [ ]:
# 1.2 搜索仓库
# gh> gh search repos "react state management" --limit 5
result = client.search_repos("react state management", per_page=5)
print(f"找到 {result.get('total_count', 0)} 个仓库 (显示 {len(result.get('items', []))} 个)\n")
for item in result.get("items", [])[:5]:
    print(f"  {item['full_name']} ⭐ {item['stargazers_count']:,} — {item.get('description', '')[:60]}")

## 2. 内容读取

验证 `GET /repos/{owner}/{repo}/contents/{path}` —— 目录列表和文件内容提取。

In [ ]:
# 2.1 列出根目录内容
# gh> gh api repos/facebook/react/contents/
contents = client.get_contents("facebook", "react", "")
print(f"根目录共 {len(contents)} 项:\n")
for item in contents[:10]:
    icon = "📁" if item['type'] == 'dir' else "📄"
    print(f"  {icon} {item['name']}")

In [ ]:
# 2.2 读取 README.md 内容
# gh> gh api repos/facebook/react/contents/README.md | jq -r .content | base64 -d
readme = client.get_file_content("facebook", "react", "README.md")
print(readme[:2000])
print(f"\n... (共 {len(readme)} 字符)")

In [ ]:
# 2.3 读取 src 目录下的文件列表
src_contents = client.get_contents("facebook", "react", "packages/react/src")
print(f"packages/react/src 共 {len(src_contents)} 项:\n")
for item in src_contents:
    print(f"  {'📁' if item['type'] == 'dir' else '📄'} {item['name']}")

## 3. Issue 操作

验证 `GET /repos/{owner}/{repo}/issues` 和 `POST /repos/{owner}/{repo}/issues`。

In [ ]:
# 3.1 列出 open issues
# gh> gh issue list --repo facebook/react --state open --limit 5
issues = client.list_issues("facebook", "react", state="open", per_page=5)
print(f"Open Issues ({len(issues)} 条):\n")
for issue in issues[:5]:
    print(f"  {format_issue(issue)}")

In [ ]:
# 3.2 获取单个 Issue 详情
# gh> gh issue view 1 --repo facebook/react
if issues:
    first = issues[0]
    detail = client.get_issue("facebook", "react", first['number'])
    print(f"\nIssue #{detail['number']}: {detail['title']}\n")
    print(detail.get('body', '无内容')[:500])
    print(f"\n  Labels: {[l['name'] for l in detail.get('labels', [])]}")
    print(f"  Comments: {detail.get('comments', 0)}")

## 4. Pull Request 操作

验证 `GET /repos/{owner}/{repo}/pulls` —— PR 列表和详情。

In [ ]:
# 4.1 列出 open PRs
# gh> gh pr list --repo facebook/react --state open --limit 5
pulls = client.list_pulls("facebook", "react", state="open", per_page=5)
print(f"Open PRs ({len(pulls)} 条):\n")
for pr in pulls[:5]:
    print(f"  {format_pull(pr)}")

In [ ]:
# 4.2 获取单个 PR 详情
# gh> gh pr view 123 --repo facebook/react
if pulls:
    first = pulls[0]
    detail = client.get_pull("facebook", "react", first['number'])
    print(f"\nPR #{detail['number']}: {detail['title']}\n")
    print(f"  Author: @{detail['user']['login']}")
    print(f"  Branch: {detail['head']['ref']} → {detail['base']['ref']}")
    print(f"  Status: {detail['state']} | Draft: {detail.get('draft', False)}")
    print(f"  Commits: {detail.get('commits', 0)} | Additions: +{detail.get('additions', 0)} | Deletions: -{detail.get('deletions', 0)}")

## 5. 代码搜索

验证 `GET /search/code` —— 跨仓库代码搜索。

In [ ]:
# 5.1 搜索代码
# gh> gh search code "useState hook" language:typescript --limit 5
result = client.search_code("useState hook language:typescript", per_page=5)
print(f"找到 {result.get('total_count', 0)} 个结果 (显示 {len(result.get('items', []))} 个)\n")
for item in result.get("items", [])[:5]:
    print(f"  📄 {item['repository']['full_name']}: {item['path']}")
    print(f"     {item['html_url']}")

In [ ]:
# 5.2 搜索 Issues
# gh> gh search issues "memory leak" repo:facebook/react --limit 5
result = client.search_issues("memory leak repo:facebook/react", per_page=5)
print(f"找到 {result.get('total_count', 0)} 个结果 (显示 {len(result.get('items', []))} 个)\n")
for item in result.get("items", [])[:5]:
    print(f"  #{item['number']} [{item['state']}] {item['title'][:60]} — @{item['user']['login']}")

## 6. 提交历史

验证 `GET /repos/{owner}/{repo}/commits` —— 提交记录和文件级历史。

In [ ]:
# 6.1 最近提交
# gh> gh api repos/facebook/react/commits?per_page=5
commits = client.get_commits("facebook", "react", per_page=5)
print(f"最近 {len(commits)} 条提交:\n")
for c in commits:
    print(f"  {format_commit(c)}")

In [ ]:
# 6.2 特定文件的提交历史
# gh> gh api repos/facebook/react/commits?path=packages/react/src/ReactHooks.js&per_page=5
file_commits = client.get_commits("facebook", "react", path="packages/react/src/ReactHooks.js", per_page=5)
print(f"packages/react/src/ReactHooks.js 最近 {len(file_commits)} 条提交:\n")
for c in file_commits:
    print(f"  {format_commit(c)}")

## 7. Actions 工作流

验证 `GET /repos/{owner}/{repo}/actions/workflows` 和运行记录。

In [ ]:
# 7.1 列出工作流
# gh> gh workflow list --repo facebook/react
workflows = client.list_workflows("facebook", "react")
print(f"工作流 ({workflows.get('total_count', 0)} 个):\n")
for wf in workflows.get("workflows", [])[:5]:
    state = "✅" if wf['state'] == 'active' else "⏸️"
    print(f"  {state} {wf['name']} (id={wf['id']}, path={wf['path']})")

In [ ]:
# 7.2 最近运行记录
# gh> gh run list --repo facebook/react --limit 5
runs = client.list_workflow_runs("facebook", "react", per_page=5)
print(f"最近运行 ({runs.get('total_count', 0)} 次):\n")
for run in runs.get("workflow_runs", [])[:5]:
    status_icon = "✅" if run['conclusion'] == 'success' else "❌" if run['conclusion'] == 'failure' else "⏳"
    print(f"  {status_icon} {run['name']} — {run['head_branch']} @ {run['run_number']}")
    print(f"     状态: {run['status']} | 结论: {run['conclusion'] or 'N/A'} | 触发: {run['event']}")
    print(f"     {run['html_url']}")

## 8. 关键发现汇总

运行完以上 Cell 后，把关键发现记录在这里：

| 能力 | API 端点 | gh CLI 等效 | 测试结果 | 备注 |
|------|---------|------------|---------|------|
| 获取仓库信息 | `GET /repos/{o}/{r}` | `gh repo view` | | |
| 搜索仓库 | `GET /search/repositories` | `gh search repos` | | |
| 列出目录 | `GET /repos/{o}/{r}/contents/{p}` | `gh api .../contents` | | |
| 读取文件 | `GET /repos/{o}/{r}/contents/{p}` | `gh api ... | base64 -d` | | 需 base64 解码 |
| 列出 Issues | `GET /repos/{o}/{r}/issues` | `gh issue list` | | |
| 获取 Issue | `GET /repos/{o}/{r}/issues/{n}` | `gh issue view` | | |
| 列出 PRs | `GET /repos/{o}/{r}/pulls` | `gh pr list` | | |
| 获取 PR | `GET /repos/{o}/{r}/pulls/{n}` | `gh pr view` | | |
| 搜索代码 | `GET /search/code` | `gh search code` | | 限 10 req/min |
| 提交历史 | `GET /repos/{o}/{r}/commits` | `gh api .../commits` | | |
| 列出工作流 | `GET /repos/{o}/{r}/actions/workflows` | `gh workflow list` | | |
| 运行记录 | `GET /repos/{o}/{r}/actions/runs` | `gh run list` | | |

### 速率限制观察

- 认证用户: 5000 req/hour
- 搜索 API: 10 req/minute（认证用户）
- 未认证: 60 req/hour

### Agent 工具设计建议

1. **必须带 GITHUB_TOKEN**：未认证速率太低（60/h），搜索 API 完全不可用
2. **前端直接调用 API**：GitHub REST API 无 CORS，前端可直接 fetch
3. **base64 解码在前端做**：`atob()` 即可解码文件内容
4. **搜索 API 独立限流**：需在前端做搜索调用频率控制
5. **Issue/PR 创建需要 write 权限**：Token 需 `repo` scope